# 第8章 面板数据与回归 —— 配套代码

对应正文 `book/part2/08-panel-regression.md`。

> **内置财务面板数据集 `fundamentals`**  
> 本 notebook 使用 `fds.load_fundamentals()` 加载内置公司-年度面板：  
> **200 家公司 × 8 年（2018—2025），共 1600 个观测**（平衡面板）。  
> 数据生成过程内置已知系数（`leverage → ROA` 真实系数 = **−0.12**）  
> 与公司固定效应（$\alpha_i$ 与 leverage **负相关**，故意制造），  
> 可直接检验 FE 能否还原真实系数、Pooled OLS 偏误有多严重。  
> 详见附录 C 数据字典。

> **提示**：先运行 `uv run python scripts/make_sample_data.py` 生成数据文件，其余格均依赖该文件。

**本 notebook 演示内容**：
1. 加载内置财务面板，展示 MultiIndex 结构
2. 描述性统计与个体效应可视化
3. Pooled OLS（混合回归）及偏误分析
4. 固定效应（FE）：个体效应 + 双向效应
5. 随机效应（RE）
6. 四种模型系数对比表
7. Hausman 检验（手动实现）
8. 普通标准误 vs 聚类标准误对比
9. 可视化：个体效应分布、拟合效果
10. 习题参考解答（可运行代码）

In [ ]:
# Cell 1：环境初始化与内置财务面板加载
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from linearmodels.panel import PanelOLS, RandomEffects, PooledOLS
from fds import load_fundamentals, set_chinese_font

set_chinese_font()

# ============================================================
# 加载内置财务面板数据集
# 平衡面板：200 家公司 × 8 年（2018—2025）= 1600 观测
# 列：firm, year, industry, roa, leverage, size, revenue_growth
# 内置已知系数：leverage -> roa 真实效应 = -0.12
# ============================================================
df_raw = load_fundamentals()

# 构造标准面板 MultiIndex（firm × year），linearmodels 要求
df = df_raw.set_index(['firm', 'year'])

# 已知真实系数（数据生成过程内置，用于验证）
BETA_LEV_TRUE = -0.12    # leverage -> ROA 真实系数
N = df.index.get_level_values('firm').nunique()
T = df.index.get_level_values('year').nunique()

print('数据集：fundamentals（内置财务面板）')
print(f'面板维度：{df.shape}')
print(f'个体数（N）：{N}，时期数（T）：{T}，总观测：{N*T}')
print(f'年份范围：{df.index.get_level_values("year").min()}—{df.index.get_level_values("year").max()}')
print('\nMultiIndex 结构（前5行）：')
df[['roa', 'leverage', 'size', 'revenue_growth']].head()

## 8.2 面板结构检验与描述性统计

检查面板是否平衡，并对核心变量做描述性统计。

In [ ]:
# Cell 2：面板结构检验与描述统计

# 平衡性检验
obs_per_firm = df.groupby('firm').size()
is_balanced = (obs_per_firm.nunique() == 1)
print(f'是否平衡面板：{is_balanced}')
print(f'每家公司观测数：均为 {obs_per_firm.iloc[0]} 年')

print('\n核心变量描述性统计：')
desc = df[['roa', 'leverage', 'size', 'revenue_growth']].describe().round(4)
desc.index.name = '统计量'
print(desc)

# 行业分布
if 'industry' in df.columns:
    ind_counts = df_raw.groupby('industry')['firm'].nunique()
    print('\n行业分布（各行业公司数）：')
    print(ind_counts.to_string())

# 组间（between）变异 vs 组内（within）变异
print('\n组间 vs 组内变异分解（std）：')
for col in ['roa', 'leverage', 'size']:
    firm_mean = df[col].groupby('firm').transform('mean')
    between_std = df[col].groupby('firm').mean().std()
    within_std  = (df[col] - firm_mean).std()
    ratio = between_std / within_std if within_std > 0 else float('nan')
    print(f'  {col:15s}: between={between_std:.4f}, within={within_std:.4f}, '
          f'组间/组内={ratio:.2f}')

## 8.3 个体效应与杠杆率的相关性可视化

`fundamentals` 数据集内置了 $\alpha_i$（个体固定效应）与 `leverage` 的**负相关**设计——  
高盈利能力（$\alpha_i$ 大）的公司倾向于保守融资（低杠杆）。  
这正是 Pooled OLS 产生偏误的根源：FE 通过组内变换可以消除该混淆。

In [ ]:
# Cell 3：公司层面均值分布 + 杠杆-ROA 散点

firm_stats = df.groupby('firm').agg(
    avg_roa=('roa', 'mean'),
    avg_lev=('leverage', 'mean'),
    avg_size=('size', 'mean'),
).reset_index()

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# 左：ROA 分布
axes[0].hist(firm_stats['avg_roa'], bins=25, color='steelblue',
             alpha=0.7, edgecolor='white')
axes[0].axvline(firm_stats['avg_roa'].mean(), color='crimson',
                lw=2, linestyle='--',
                label=f'均值={firm_stats["avg_roa"].mean():.3f}')
axes[0].set_title('公司平均 ROA 分布', fontsize=11)
axes[0].set_xlabel('平均 ROA')
axes[0].set_ylabel('公司数量')
axes[0].legend()

# 中：杠杆率分布
axes[1].hist(firm_stats['avg_lev'], bins=25, color='darkorange',
             alpha=0.7, edgecolor='white')
axes[1].axvline(firm_stats['avg_lev'].mean(), color='crimson',
                lw=2, linestyle='--',
                label=f'均值={firm_stats["avg_lev"].mean():.3f}')
axes[1].set_title('公司平均杠杆率分布', fontsize=11)
axes[1].set_xlabel('平均 leverage')
axes[1].set_ylabel('公司数量')
axes[1].legend()

# 右：平均杠杆 vs 平均ROA（公司层面截面）
axes[2].scatter(firm_stats['avg_lev'], firm_stats['avg_roa'],
                alpha=0.5, s=30, color='seagreen')
rho_cross = np.corrcoef(firm_stats['avg_lev'], firm_stats['avg_roa'])[0, 1]
axes[2].set_title(f'公司均值：杠杆 vs ROA\n截面相关 ρ = {rho_cross:.3f}', fontsize=11)
axes[2].set_xlabel('公司平均杠杆率')
axes[2].set_ylabel('公司平均 ROA')

plt.suptitle('内置财务面板 fundamentals —— 公司层面描述', y=1.02, fontsize=12)
plt.tight_layout()
plt.show()

print(f'截面（公司均值）层面：杠杆 vs ROA 相关系数 = {rho_cross:.3f}')
print('→ 截面层面负相关较强，Pooled OLS 会把个体效应混淆进杠杆系数，导致负偏误被夸大')

## 8.4 混合 OLS（Pooled OLS）

忽略面板结构，把所有 1600 个观测当截面数据处理。  
预期：`leverage` 系数因个体效应混淆而**严重偏离真实值 −0.12**，  
约为 **−0.40**，偏误达 −0.28（夸大了三倍以上的负效应）。

> **内置真实系数**：$\beta_{\text{leverage}} = -0.12$

In [ ]:
# Cell 4：Pooled OLS

y = df['roa']
X = sm.add_constant(df[['leverage', 'size']])

res_pooled = PooledOLS(y, X).fit()

print('=' * 60)
print('Pooled OLS 估计结果（忽略面板结构）')
print('=' * 60)
print(res_pooled.summary.tables[1])
print(f'\nR²: {res_pooled.rsquared:.4f}')

coef_pooled = res_pooled.params
print('\n=== 与内置真实系数对比 ===')
print(f'leverage 系数: 估计={coef_pooled["leverage"]:+.4f}, '
      f'真实={BETA_LEV_TRUE:+.4f}, '
      f'偏误={coef_pooled["leverage"]-BETA_LEV_TRUE:+.4f}')
print('\n→ Pooled OLS 大幅夸大了杠杆的负效应：')
print(f'  估计值 ≈ {coef_pooled["leverage"]:.2f}，'
      f'是真实值 {BETA_LEV_TRUE} 的 '
      f'{abs(coef_pooled["leverage"]/BETA_LEV_TRUE):.1f} 倍')
print('  根源：alpha_i（盈利能力）与 leverage 负相关，Pooled OLS 把它混入了杠杆系数')

## 8.5 固定效应模型（FE）

### 8.5.1 个体固定效应

组内变换消去 $\alpha_i$，只利用公司内部时间变化识别系数。  
预期：`leverage` 系数 ≈ **−0.13**，接近内置真实值 −0.12，FE 成功消除偏误。

In [ ]:
# Cell 5：个体固定效应（FE）—— 普通SE vs 聚类SE

X_fe = df[['leverage', 'size']]

# 5a. 普通（同方差）标准误
res_fe_plain = PanelOLS(y, X_fe, entity_effects=True).fit(cov_type='unadjusted')

# 5b. 聚类标准误（按公司）
res_fe_cl = PanelOLS(y, X_fe, entity_effects=True).fit(
    cov_type='clustered', cluster_entity=True
)

print('=' * 60)
print('个体固定效应（FE）—— 聚类标准误（按公司）')
print('=' * 60)
print(res_fe_cl.summary.tables[1])
print(f'\nR²（组内）: {res_fe_cl.rsquared:.4f}')

coef_fe = res_fe_cl.params
print('\n=== 与内置真实系数对比 ===')
print(f'leverage: 估计={coef_fe["leverage"]:+.4f}, 真实={BETA_LEV_TRUE:+.4f}, '
      f'偏误={coef_fe["leverage"]-BETA_LEV_TRUE:+.4f}')
print('→ FE 系数 ≈ -0.13，接近真实值 -0.12，成功消除个体效应混淆 ✓')

print('\n=== 普通SE vs 聚类SE 对比 ===')
se_plain = res_fe_plain.std_errors
se_cl    = res_fe_cl.std_errors
for v in ['leverage', 'size']:
    if v in se_plain.index:
        ratio = se_cl[v] / se_plain[v] if se_plain[v] > 0 else float('nan')
        print(f'{v:15s}: 普通SE={se_plain[v]:.5f}, 聚类SE={se_cl[v]:.5f}, '
              f'倍数={ratio:.2f}x')

### 8.5.2 双向固定效应（Two-Way FE）

同时控制个体效应（公司特征）和时间效应（宏观年度冲击）。

In [ ]:
# Cell 6：双向固定效应（个体 + 时间）

res_fe2w = PanelOLS(y, X_fe, entity_effects=True, time_effects=True).fit(
    cov_type='clustered', cluster_entity=True
)

print('=' * 60)
print('双向固定效应（个体 + 时间）—— 聚类标准误')
print('=' * 60)
print(res_fe2w.summary.tables[1])
print(f'\nR²（组内）: {res_fe2w.rsquared:.4f}')

coef_fe2w = res_fe2w.params
print('\n=== 与内置真实系数对比 ===')
print(f'leverage: 估计={coef_fe2w["leverage"]:+.4f}, 真实={BETA_LEV_TRUE:+.4f}')
print('→ 控制时间效应后进一步消除宏观混淆，系数更接近真实值')

## 8.6 随机效应模型（RE）

将 $\alpha_i$ 视为随机误差的一部分，用 GLS 估计。  
效率比 FE 高，但要求 $\alpha_i$ 与 $X$ 不相关——  
`fundamentals` 数据集故意违反此假设（$\alpha_i$ 与 leverage 负相关），  
故 RE 系数会介于 Pooled OLS 与 FE 之间（有偏）。

In [ ]:
# Cell 7：随机效应（RE）

X_re = sm.add_constant(df[['leverage', 'size']])
res_re = RandomEffects(y, X_re).fit()

print('=' * 60)
print('随机效应（RE）估计结果')
print('=' * 60)
print(res_re.summary.tables[1])

coef_re = res_re.params
print('\n=== 与内置真实系数对比 ===')
print(f'leverage: 估计={coef_re["leverage"]:+.4f}, 真实={BETA_LEV_TRUE:+.4f}')
print('\n→ RE 系数介于 Pooled OLS（夸大负效应）和 FE（接近真实）之间')
print('  因数据集设计上 alpha_i 与 X 相关，RE 假设被违反，有偏')

## 8.7 四种模型系数对比表

一张表汇总 Pooled OLS / FE(个体) / FE(双向) / RE 的系数、标准误与对内置真实值的偏误。

**教学亮点**：  
- Pooled OLS `leverage` 系数 ≈ −0.40，**严重夸大**负效应（比真实值 −0.12 大约 3 倍）  
- FE（个体）系数 ≈ −0.13，**接近真实值 −0.12**，FE 消除偏误成功

In [ ]:
# Cell 8：四种模型系数对比表

models = {
    'Pooled OLS':    res_pooled,
    'FE（个体）':    res_fe_cl,
    'FE（双向）':    res_fe2w,
    'RE':            res_re,
}

rows_cmp = []
for name, res in models.items():
    p  = res.params
    se = res.std_errors
    rows_cmp.append({
        '模型':        name,
        'β_leverage': round(p['leverage'], 4),
        'SE(lev)':    round(se['leverage'], 4),
        '偏误(lev)':  round(p['leverage'] - BETA_LEV_TRUE, 4),
        'β_size':     round(p['size'], 4),
        'SE(size)':   round(se['size'], 4),
    })

df_cmp = pd.DataFrame(rows_cmp).set_index('模型')

print(f'四种模型系数对比（内置真实系数：leverage={BETA_LEV_TRUE}）')
print('=' * 75)
print(df_cmp.to_string())
print('=' * 75)
print()
print('核心结论：')
lev_pool = res_pooled.params['leverage']
lev_fe   = res_fe_cl.params['leverage']
lev_re   = res_re.params['leverage']
print(f'  Pooled OLS leverage ≈ {lev_pool:.2f}  ->  偏误 = {lev_pool-BETA_LEV_TRUE:+.2f}')
print(f'  FE（个体）leverage  ≈ {lev_fe:.2f}  ->  偏误 = {lev_fe-BETA_LEV_TRUE:+.2f}  (接近真实值)')
print(f'  RE           leverage  ≈ {lev_re:.2f}  ->  偏误 = {lev_re-BETA_LEV_TRUE:+.2f}')
print(f'  真实值：{BETA_LEV_TRUE}')

## 8.8 Hausman 检验

检验 $H_0$：$\text{Cov}(\mathbf{x}_{it}, \alpha_i) = 0$（RE 假设成立）  

`fundamentals` 数据集内置 $\alpha_i$ 与 leverage 负相关，  
期望 Hausman 检验**拒绝 $H_0$**（p < 0.05），从而支持选用 FE。

In [ ]:
# Cell 9：Hausman 检验（手动实现）
from scipy import stats as scipy_stats

# 使用同方差FE（Hausman检验基于传统SE）
res_fe_plain_full = PanelOLS(y, X_fe, entity_effects=True).fit(cov_type='unadjusted')
res_re_full = RandomEffects(y, X_re).fit()

common_vars = ['leverage', 'size']

b_fe = res_fe_plain_full.params[common_vars].values
b_re = res_re_full.params[common_vars].values

V_fe = res_fe_plain_full.cov[common_vars].loc[common_vars].values
V_re = res_re_full.cov[common_vars].loc[common_vars].values

diff = b_fe - b_re
V_diff = V_fe - V_re

try:
    V_diff_inv = np.linalg.inv(V_diff)
    H_stat = float(diff @ V_diff_inv @ diff)
    k = len(common_vars)
    p_val = 1 - scipy_stats.chi2.cdf(H_stat, df=k)
    crit = scipy_stats.chi2.ppf(0.95, df=k)
    print('Hausman 检验结果（手动实现）')
    print('=' * 55)
    print(f'统计量 H  = {H_stat:.4f}')
    print(f'自由度 k  = {k}')
    print(f'p 值      = {p_val:.6f}')
    print(f'5%临界值  = {crit:.4f}')
    print()
    if p_val < 0.05:
        print('结论：拒绝 H0（p < 0.05）-> alpha_i 与 X 相关 -> 选择 FE')
        print('      与数据集设计一致（alpha_i 与 leverage 刻意设为负相关）')
    else:
        print('结论：不能拒绝 H0（p >= 0.05）-> 可考虑效率更高的 RE')
except np.linalg.LinAlgError:
    print('V_diff 不可逆，使用广义逆')
    H_stat = float(diff @ np.linalg.pinv(V_diff) @ diff)
    p_val = 1 - scipy_stats.chi2.cdf(H_stat, df=len(common_vars))
    print(f'H（广义逆）= {H_stat:.4f}, p = {p_val:.6f}')

print()
print('系数差异明细：')
for v, d, bfe, bre in zip(common_vars, diff, b_fe, b_re):
    print(f'  {v:12s}: FE={bfe:+.4f}, RE={bre:+.4f}, 差={d:+.4f}')

## 8.9 聚类标准误的重要性：可视化对比

同样的 FE 模型，普通标准误 vs 聚类标准误，差异有多大？  
金融面板存在序列相关（同公司跨期相关），聚类 SE 更保守（更诚实）。

In [ ]:
# Cell 10：标准误对比可视化（普通 vs 聚类）

ci_data = []
for label, res in [
    ('FE 普通SE',     res_fe_plain),
    ('FE 聚类SE',     res_fe_cl),
    ('FE双向 聚类SE', res_fe2w),
    ('Pooled OLS',   res_pooled),
    ('RE',           res_re),
]:
    for var in ['leverage', 'size']:
        if var in res.params.index:
            coef = res.params[var]
            se   = res.std_errors[var]
            ci_data.append({
                '模型': label, '变量': var,
                '系数': coef, 'SE': se,
                'CI下': coef - 1.96 * se,
                'CI上': coef + 1.96 * se,
            })

df_ci = pd.DataFrame(ci_data)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
model_order = ['Pooled OLS', 'FE 普通SE', 'FE 聚类SE', 'FE双向 聚类SE', 'RE']
colors = ['#d62728', '#1f77b4', '#2ca02c', '#17becf', '#ff7f0e']

for ax, var, true_val, title in [
    (axes[0], 'leverage', BETA_LEV_TRUE,
     f'leverage 系数（内置真实值={BETA_LEV_TRUE}）'),
    (axes[1], 'size', None, 'size 系数'),
]:
    sub = df_ci[df_ci['变量'] == var].copy()
    sub = sub.set_index('模型').loc[model_order]
    y_pos = range(len(model_order))
    for i, (m, row) in enumerate(sub.iterrows()):
        ax.errorbar(row['系数'], i,
                    xerr=[[row['系数'] - row['CI下']], [row['CI上'] - row['系数']]],
                    fmt='o', color=colors[i], capsize=5, markersize=7,
                    label=m, elinewidth=2)
    if true_val is not None:
        ax.axvline(true_val, color='black', lw=2, linestyle='--',
                   label=f'真实值={true_val}')
    ax.set_yticks(list(y_pos))
    ax.set_yticklabels(model_order)
    ax.set_title(title, fontsize=11)
    ax.set_xlabel('系数（95% CI）')
    ax.legend(fontsize=8, loc='upper right')
    ax.grid(axis='x', alpha=0.3)

plt.suptitle('各模型系数估计与置信区间（内置财务面板 fundamentals）',
             fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

print('观察：')
print('  1. Pooled OLS 系数（真实值虚线左侧）偏离最远，严重夸大负效应')
print('  2. FE 系数（聚类SE）覆盖真实值 -0.12，估计准确')
print('  3. FE 普通SE 置信区间最窄（低估不确定性），聚类SE 更保守诚实')

## 8.10 FE 拟合值与残差分析

In [ ]:
# Cell 11：拟合值与残差可视化

fitted    = res_fe_cl.fitted_values.squeeze()
residuals = res_fe_cl.resids.squeeze()

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# 左：实际值 vs 拟合值
axes[0].scatter(fitted, y, alpha=0.3, s=10, color='steelblue')
lim_min = min(fitted.min(), y.min()) * 0.95
lim_max = max(fitted.max(), y.max()) * 1.05
axes[0].plot([lim_min, lim_max], [lim_min, lim_max], 'r--', lw=1.5, label='45度线')
axes[0].set_xlabel('FE 拟合值')
axes[0].set_ylabel('实际 ROA')
axes[0].set_title('实际值 vs FE 拟合值')
axes[0].legend()

# 中：残差分布
axes[1].hist(residuals, bins=40, color='seagreen', alpha=0.7,
             edgecolor='white', density=True)
axes[1].set_title('FE 残差分布')
axes[1].set_xlabel('残差')
axes[1].set_ylabel('概率密度')

# 右：残差 vs 拟合值
axes[2].scatter(fitted, residuals, alpha=0.3, s=10, color='darkorange')
axes[2].axhline(0, color='black', lw=1.5, linestyle='--')
axes[2].set_xlabel('FE 拟合值')
axes[2].set_ylabel('残差')
axes[2].set_title('残差 vs 拟合值（无明显模式则良好）')

plt.suptitle('FE（个体效应，聚类SE）拟合诊断 —— fundamentals 数据集',
             fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

## 8.11 FE 估计的个体效应还原

固定效应估计器可以还原各公司的相对个体效应 $\hat{\alpha}_i$。  
由于 `fundamentals` 数据集的 $\alpha_i$ 与 leverage 负相关，  
验证 FE 估计出的公司均值残差是否与 leverage 负相关——  
这正是 Hausman 检验拒绝 $H_0$ 的直觉来源。

In [ ]:
# Cell 12：估计个体效应并验证其与杠杆率的相关性

beta_hat = res_fe_cl.params.values  # [leverage, size]
X_vals   = df[['leverage', 'size']].values

df_est = df[['roa', 'leverage', 'size']].copy()
df_est['xbeta']     = X_vals @ beta_hat
df_est['alpha_hat'] = df_est['roa'] - df_est['xbeta']

alpha_hat_firm = df_est.groupby('firm')['alpha_hat'].mean()
alpha_hat_c    = alpha_hat_firm - alpha_hat_firm.mean()

avg_lev_firm = df['leverage'].groupby('firm').mean()

rho_alpha_lev = np.corrcoef(alpha_hat_c.values, avg_lev_firm.values)[0, 1]

fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(avg_lev_firm.values, alpha_hat_c.values,
           alpha=0.5, s=30, color='steelblue')
ax.axhline(0, color='black', lw=1, linestyle='--')
ax.set_xlabel('公司平均杠杆率 leverage')
ax.set_ylabel('FE 估计的公司效应 $\\hat{\\alpha}_i$（中心化）')
ax.set_title(
    f'FE 估计个体效应 vs 平均杠杆率\n相关系数 ρ = {rho_alpha_lev:.3f}',
    fontsize=11
)
plt.tight_layout()
plt.show()

print(f'FE 估计的 alpha_i vs 公司平均杠杆 相关系数：{rho_alpha_lev:.3f}')
print('→ 负相关，与数据生成设计一致：高盈利能力公司低杠杆')
print('→ 这正是 Pooled OLS 产生偏误的根源（alpha_i 与 X 相关）')

## 8.12 本章小结

| 模型 | leverage 系数 | 偏误 | 备注 |
|------|--------------|------|------|
| Pooled OLS | ≈ −0.40 | **−0.28（严重夸大）** | alpha_i 混入误差项 |
| FE（个体） | ≈ −0.13 | ≈ −0.01（接近0） | FE 消除偏误 |
| FE（双向） | ≈ −0.13 | ≈ −0.01 | 额外控制时间效应 |
| RE | 介于两者之间 | 有偏 | 假设被违反 |

**黄金法则**：金融面板研究 = FE（个体或双向）+ 聚类稳健标准误（按公司）

## 8.13 习题参考解答

以下代码对应正文第8.11节习题，使用 `load_fundamentals()` 可直接运行。

In [ ]:
# === 习题8.1：面板结构识别 ===
print('=' * 60)
print('习题 8.1：面板结构识别')
print('=' * 60)

obs_per_firm = df.groupby('firm').size()
is_bal = (obs_per_firm.nunique() == 1)
print(f'(a) 是否平衡面板：{is_bal}')
print(f'    每家公司观测数：min={obs_per_firm.min()}, max={obs_per_firm.max()}')

print(f'(b) MultiIndex 层级：{df.index.names}')
print(f'    总观测数：{len(df)}，= {N} 公司 × {T} 年')

print('(c) 每家公司观测年数分布：')
print(obs_per_firm.value_counts().sort_index().to_dict())

rng_ex = np.random.default_rng(99)
mask = rng_ex.random(len(df)) > 0.15
df_unbal = df[mask].copy()
obs_unbal = df_unbal.groupby('firm').size()
print(f'\n模拟非平衡面板：{len(df_unbal)} 观测，'
      f'每公司年数范围 [{obs_unbal.min()}, {obs_unbal.max()}]')

In [ ]:
# === 习题8.2：FE 组内变换推导（数值验证）===
print('=' * 60)
print('习题 8.2：FE 组内变换数值验证')
print('=' * 60)

demo_firms = df.index.get_level_values('firm').unique()[:3].tolist()
df_demo = df.loc[demo_firms]

df_demo_within = df_demo[['roa', 'leverage', 'size']].copy()
for col in ['roa', 'leverage', 'size']:
    mean_col = df_demo_within[col].groupby('firm').transform('mean')
    df_demo_within[f'{col}_within'] = df_demo_within[col] - mean_col

print('(a) 组内变换后各公司均值（应均为0）：')
print(df_demo_within[['roa_within', 'leverage_within', 'size_within']]
      .groupby('firm').mean().round(10))

df_demo_within['const_x'] = 1.0
mean_const = df_demo_within['const_x'].groupby('firm').transform('mean')
df_demo_within['const_within'] = df_demo_within['const_x'] - mean_const
print(f'\n(c) 不随时间变化变量经组内变换后方差：'
      f'{df_demo_within["const_within"].var():.2e}')
print('  → 完全消失，FE 无法估计不随时间变化的效应')

In [ ]:
# === 习题8.3：Pooled OLS 偏误方向分析 ===
print('=' * 60)
print('习题 8.3：Pooled OLS 偏误方向分析（基于 fundamentals）')
print('=' * 60)

b_pool   = res_pooled.params['leverage']
b_fe_val = res_fe_cl.params['leverage']

print('(a) 偏误方向预测：')
print('    alpha_i 与 leverage 负相关（高能力 -> 低杠杆）')
print('    alpha_i 对 ROA 正影响')
print('    -> Pooled OLS 高估杠杆的负效应（|β̂_OLS| >> |β_true|）')
print()
print('(b) 数值验证：')
print(f'    Pooled OLS β_leverage = {b_pool:+.4f}')
print(f'    FE β_leverage         = {b_fe_val:+.4f}')
print(f'    真实值（内置）         = {BETA_LEV_TRUE:+.4f}')
print()
print('(c) 偏误大小：')
pct = abs((b_pool - BETA_LEV_TRUE) / BETA_LEV_TRUE) * 100
print(f'    Pooled OLS - 真实值 = {b_pool - BETA_LEV_TRUE:+.4f}  （夸大约 {pct:.0f}%）')
print(f'    FE - 真实值         = {b_fe_val - BETA_LEV_TRUE:+.4f}  （FE 几乎无偏）')

In [ ]:
# === 习题8.4：手动 Hausman 检验（单变量版）===
print('=' * 60)
print('习题 8.4：手动 Hausman 检验（leverage 单变量）')
print('=' * 60)

b_fe_lev   = res_fe_plain_full.params['leverage']
b_re_lev   = res_re_full.params['leverage']
var_fe_lev = res_fe_plain_full.cov.loc['leverage', 'leverage']
var_re_lev = res_re_full.cov.loc['leverage', 'leverage']

diff_lev     = b_fe_lev - b_re_lev
var_diff_lev = var_fe_lev - var_re_lev

print(f'β_FE  = {b_fe_lev:.6f},  Var(β_FE)  = {var_fe_lev:.8f}')
print(f'β_RE  = {b_re_lev:.6f},  Var(β_RE)  = {var_re_lev:.8f}')
print(f'差值  = {diff_lev:.6f},  Var差      = {var_diff_lev:.8f}')

if var_diff_lev > 0:
    H_manual = diff_lev**2 / var_diff_lev
    p_manual = 1 - scipy_stats.chi2.cdf(H_manual, df=1)
    crit_val = scipy_stats.chi2.ppf(0.95, df=1)
    print(f'\nH = {H_manual:.4f}')
    print(f'χ²(1) 5%临界值 = {crit_val:.4f}')
    print(f'p 值 = {p_manual:.6f}')
    if p_manual < 0.05:
        print('结论：拒绝H0 -> 选FE（与全变量Hausman一致）')
    else:
        print('结论：不拒绝H0 -> FE与RE系数无显著差异')
else:
    print('Var差 <= 0，建议使用全变量版 Hausman 检验（Cell 9）')

In [ ]:
# === 习题8.5：双向FE vs 单向FE 时间效应检验 ===
print('=' * 60)
print('习题 8.5：双向FE vs 单向FE 时间效应显著性')
print('=' * 60)

print('(a) 单向FE vs 双向FE 系数对比：')
for var in ['leverage', 'size']:
    b1 = res_fe_cl.params[var]
    b2 = res_fe2w.params[var]
    print(f'  {var:10s}: 单向FE={b1:+.4f}, 双向FE={b2:+.4f}, 差={b2-b1:+.4f}')

print()
print('(b) 为何时间效应影响估计：')
print('    宏观经济冲击（利率、GDP增速）同时影响所有公司ROA与杠杆决策。')
print('    不控制时间效应会把宏观趋势错误归因到杠杆系数上。')

print()
print('(c) 双向FE摘要（含F统计量）：')
print(res_fe2w.summary.tables[0])

# 用年度虚拟变量辅助回归验证时间效应
df_aug = df.reset_index().copy()
year_dummies = pd.get_dummies(df_aug['year'], prefix='yr', drop_first=True).astype(float)
df_aug = pd.concat([df_aug, year_dummies], axis=1).set_index(['firm', 'year'])

yd_cols = [c for c in df_aug.columns if c.startswith('yr_')]
X_aug = df_aug[['leverage', 'size'] + yd_cols]
res_aug = PanelOLS(df_aug['roa'], X_aug, entity_effects=True).fit(
    cov_type='clustered', cluster_entity=True
)

print('\n年度虚拟变量系数（双向FE，以最早年份为基准）：')
yr_coefs = res_aug.params[yd_cols]
yr_pvals = res_aug.pvalues[yd_cols]
yr_df = pd.DataFrame({'系数': yr_coefs.round(4), 'p值': yr_pvals.round(4)})
yr_df.index = [c.replace('yr_', '') for c in yd_cols]
yr_df.index.name = '年份'
print(yr_df.to_string())